# Episode 3: Tools

An agent that can only talk is like an employee who can only attend meetings. **Tools** let an agent *act* — call an API, query a database, run a calculation.

**In this episode you'll build:** an agent that answers weather questions by calling functions you write.

## Why tools?

An LLM is a great reasoner trapped in a text box. It can't look up today's weather, query your database, or do reliable arithmetic on demand.

A **tool** hands the model a capability. The model decides *when* to call it and *with what arguments* — you just supply the function. In the Beta API a tool is, at its simplest, **a plain Python function**.

## Step 1: A tool is a plain function

Two things make a function usable as a tool:

- **Type annotations** — the model uses them to build the call (`city: str`)
- **A docstring** — the model uses it to decide *when* to call the tool

That's all. No registration, no decorator required.

In [ ]:
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # A real tool would call a weather API. We simulate for the workshop.
    return f"{city}: 22 degrees C, sunny."

## Step 2: The `@tool` decorator (optional)

Passing a plain function works. The `@tool` decorator is for when you want to **customize** the tool — override its name or description, or attach middleware. It's optional, but good to recognize.

Here's a second tool, written with the decorator.

In [ ]:
from autogen.beta import tool


@tool
def convert_celsius(celsius: float) -> str:
    """Convert a temperature in Celsius to Fahrenheit."""
    return f"{celsius} C = {celsius * 9 / 5 + 32} F"

## Step 3: Give the agent its tools

Pass the functions in the `tools=[...]` list. The agent can now call either one — and decide which, on its own.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

weather_agent = Agent(
    "weather_agent",
    prompt="Use your tools to answer weather questions.",
    config=config,
    tools=[get_weather, convert_celsius],
)

## Step 4: Ask — the agent picks the tools

Notice we never tell the agent *which* tool to use. It reads the question, calls `get_weather`, then `convert_celsius`, and combines the results.

In [1]:
reply = await weather_agent.ask(
    "What is the weather in Tokyo, and what is that in Fahrenheit?"
)
print(reply.body)

The weather in Tokyo is currently 22 degrees Celsius and sunny. That temperature is equivalent to 71.6 degrees Fahrenheit.


## Builtin tools

You don't have to write every tool. AG2 Beta ships ready-made tools in `autogen.beta.tools` — web search, web fetch, code execution, a shell, MCP servers, and more. You construct one and drop it straight into `tools=[...]`.

In [2]:
from autogen.beta.tools import WebSearchTool, WebFetchTool

# Builtin tools are constructed, then added to an agent like any other tool.
search = WebSearchTool()
fetch = WebFetchTool()
print("Ready:", type(search).__name__, "+", type(fetch).__name__)

Ready: WebSearchTool + WebFetchTool


## Additional content

**The model decides.** You give an agent a *set* of tools; the model chooses which to call, in what order, and with what arguments — based on the docstrings and type hints. Clear docstrings are not optional polish; they are how the model routes.

**Tools can return more than strings.** A tool can return a Pydantic model or a file. The agent handles the type correctly — useful for structured results (Episode 9).

**Builtin tools** (Episode 19 goes deeper): `WebSearchTool`, `WebFetchTool`, `CodeExecutionTool`, `LocalShellTool`, `MCPServerTool`, and others — all in `autogen.beta.tools`.

## What's next

Your agent can think and act. But one agent still means one perspective. Next, you'll have one agent **delegate** to others — using a sub-agent as a tool.